<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Gömme Katmanları ile Doğrusal Katmanlar Arasındaki Farkı Anlamak

- PyTorch'taki gömme (embedding) katmanları, matris çarpımı yapan doğrusal katmanlarla aynı işi görür; gömme katmanlarını kullanmamızın nedeni hesaplama verimliliğidir
- Bu ilişkiyi PyTorch kod örnekleriyle adım adım inceleyeceğiz

In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.3.1


<br>
&nbsp;

## nn.Embedding kullanımı

In [2]:
# Elimizde şu 3 eğitim örneği olduğunu varsayalım;
# bunlar bir LLM bağlamında token kimliklerini temsil edebilir
idx = torch.tensor([2, 3, 1])

# Gömme matrisindeki satır sayısı, en büyük token kimliği + 1
# alınarak belirlenebilir.
# En yüksek token kimliği 3 ise, olası 0, 1, 2, 3 token kimlikleri için
# 4 satır isteriz
num_idx = max(idx)+1

# İstenen gömme boyutu bir hiperparametredir
out_dim = 5

- Basit bir gömme katmanı uygulayalım:

In [3]:
# Gömme katmanındaki ağırlıklar küçük rastgele değerlerle
# başlatıldığı için, sonuçların yeniden üretilebilir olması adına
# rastgele tohum (seed) kullanıyoruz
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

İstersek gömme ağırlıklarına bakabiliriz:

In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

- Ardından, kimliği 1 olan bir eğitim örneğinin vektör temsilini elde etmek için gömme katmanını kullanabiliriz:

In [5]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Aşağıda, arka planda neler olduğunun görselleştirmesi yer alıyor:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/1.png" width="400px">

- Benzer şekilde, kimliği 2 olan bir eğitim örneğinin vektör temsilini elde etmek için de gömme katmanlarını kullanabiliriz:

In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/2.png" width="400px">

- Şimdi daha önce tanımladığımız tüm eğitim örneklerini dönüştürelim:

In [7]:
idx = torch.tensor([2, 3, 1])
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Arka planda yine aynı arama (look-up) mantığı işliyor:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/3.png" width="450px">

<br>
&nbsp;

## nn.Linear kullanımı

- Şimdi, yukarıdaki gömme katmanının PyTorch'ta one-hot kodlanmış bir temsile uygulanan `nn.Linear` katmanıyla tam olarak aynı işi yaptığını göstereceğiz
- Önce token kimliklerini one-hot temsile dönüştürelim:

In [8]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

- Ardından, $X W^\top$ matris çarpımını gerçekleştiren bir `Linear` katmanı başlatıyoruz:

In [9]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

- PyTorch'taki doğrusal katmanın da küçük rastgele ağırlıklarla başlatıldığını unutmayın; yukarıdaki `Embedding` katmanıyla doğrudan karşılaştırabilmek için aynı küçük rastgele ağırlıkları kullanmamız gerekir, bu yüzden burada onları yeniden atıyoruz:

In [10]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

- Şimdi doğrusal katmanı, girdilerin one-hot kodlanmış temsili üzerinde kullanabiliriz:

In [11]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

Görüldüğü gibi bu, gömme katmanını kullandığımızda elde ettiğimizin tamamen aynısı:

In [12]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Arka planda, ilk eğitim örneğinin token kimliği için şu hesaplama yapılır:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/4.png" width="450px">

- Ve ikinci eğitim örneğinin token kimliği için:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/5.png" width="450px">

- Her one-hot kodlanmış satırda (tasarım gereği) bir indeks dışındaki tüm değerler 0 olduğundan, bu matris çarpımı esasen one-hot elemanlarının aranmasıyla aynı şeydir
- One-hot kodlamalar üzerinde matris çarpımı kullanmak, gömme katmanı aramasına eşdeğerdir; ancak büyük gömme matrisleriyle çalışırken verimsiz olabilir, çünkü sıfırla yapılan pek çok gereksiz çarpım vardır